# 02 · Дообучение и оценка объектной сегментации

**Результат:** корректно подготовленные маски, три блока вашей реализации,
два пилота и сравнение Mask R-CNN с YOLO11n-seg после дообучения.
Старт на занятии, длинные запуски и отчёт — самостоятельно. Сохраните выполненный ноутбук,
CSV/JSON с результатами, кривые и 2–3 общих изображения с разбором ошибок.

Объект этого опыта — специализация моделей на небольшом наборе пешеходов.
Равное число эпох не делает все условия одинаковыми: различаются размер и устройство
сетей, обработка изображения, нормализация и форматы обучающих масок.

## Среда и данные

Запускайте ячейки сверху вниз. Для первого запуска нужен интернет: Penn–Fudan ≈52 MiB,
веса Mask R-CNN ≈170 MiB и YOLO11n-seg ≈6 MiB. Код содержит все функции и работает
без соседних файлов. Загрузки проверяются по SHA-256. Веса и результаты сохраняются
в `segmentation_lab_data`, которую не нужно сдавать целиком.

Локально используйте окружение предыдущей работы. В Colab загрузите этот `.ipynb`
через **Файл → Открыть блокнот → Загрузить**. Для обучения включите GPU до импортов.
Демо работает на CPU. Размер входа здесь учебный, 320 пикселей; Mask R-CNN сохраняет
пропорции с ограничением длинной стороны 512, YOLO использует letterbox.

Данные: 170 изображений, 423 размеченных человека; фиксированный split **120 / 25 / 25**, seed 42.
Это знакомый класс `person`, уже присутствующий в COCO, а не обучение новой категории.
ID экземпляра имеет смысл внутри одного изображения и не является ID человека между кадрами.
Некоторые фоновые люди не размечены. FP означает ошибку относительно доступной разметки;
в отчёте отличайте её неполноту от ошибки модели. Split сделан по изображениям, а не независимым сценам.

In [ ]:
# Сохраняем установленную пару PyTorch/torchvision, включая сборку CUDA.
import importlib.metadata as meta
import subprocess, sys
packages=['ultralytics==8.4.102','numpy>=1.26,<3','pandas>=2.1,<4',
          'matplotlib>=3.8,<4','Pillow>=10,<13','ipywidgets>=8.1,<9']
try:
    ready=meta.version('ultralytics')=='8.4.102'
    for name in ['torch','torchvision','numpy','pandas','matplotlib','Pillow','ipywidgets']:meta.version(name)
except meta.PackageNotFoundError:ready=False
if not ready:
    for name in ['torch','torchvision']:
        try:packages.append(name+'=='+meta.version(name))
        except meta.PackageNotFoundError:pass
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*packages])
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:pass
print('Библиотеки готовы. Если они уже были импортированы до установки, перезапустите ядро.')

In [ ]:
"""Shared cells embedded into the distributed notebooks by build_segmentation.py."""
from pathlib import Path
import os, json, time, hashlib, urllib.request, shutil, zipfile, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch
import torchvision
import cv2
import ultralytics
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def fetch(url, path, expected):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256_file(path) == expected:
        return path
    part = path.with_suffix(path.suffix + '.part')
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'SegmentationLab/1.0'})
        with urllib.request.urlopen(request, timeout=90) as source, part.open('wb') as out:
            shutil.copyfileobj(source, out)
        if sha256_file(part) != expected:
            raise ValueError('Контрольная сумма не совпала: ' + path.name)
        part.replace(path)
    except Exception:
        part.unlink(missing_ok=True)
        raise
    return path


WEIGHT_SOURCES = {
    'maskrcnn': ('https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth',
                'maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth',
                'bf2d0c1efbc936eeee2bc95a48e80ebc86b891f61b0106485937fc29f9315fc0'),
    'yolo11n_seg': ('https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n-seg.pt',
                    'yolo11n-seg.pt',
                    '55ed65c56c91713d23e8402371c6c49a6fd84f257f7dce452e8d70e41dcbe152'),
}
DATA_SHA = '9095a9613c95586f1c7f2a327d454833d16e0f5e17e5f83d35027ffd315b48e2'


def prepare_data(root):
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    archive = fetch('https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip', root/'PennFudanPed.zip', DATA_SHA)
    data = root/'PennFudanPed'
    if not (data/'.segmentation_extracted').exists():
        with zipfile.ZipFile(archive) as z:
            for m in z.infolist():
                if not (root/m.filename).resolve().is_relative_to(root.resolve()):
                    raise ValueError('Некорректный путь в архиве')
            z.extractall(root)
        (data/'.segmentation_extracted').touch()
    paths = sorted((data/'PNGImages').glob('*.png'))
    assert len(paths) == 170
    items = [{'name':p.name, 'image':p, 'mask':data/'PedMasks'/(p.stem+'_mask.png')} for p in paths]
    order = np.random.default_rng(42).permutation(len(items))
    split = {k:[items[i] for i in ix] for k,ix in [('train',order[:120]),('val',order[120:145]),('test',order[145:])]}
    manifest = {k:[s['name'] for s in v] for k,v in split.items()}
    (root/'split_seed42.json').write_text(json.dumps(manifest, indent=2))
    assert len(set(sum(manifest.values(), []))) == 170
    return split


def read_sample(item):
    with Image.open(item['image']) as im: image = np.array(im.convert('RGB'))
    with Image.open(item['mask']) as im: labels = np.array(im)
    return image, labels


def weight_path(key):
    url, name, sha = WEIGHT_SOURCES[key]
    return fetch(url, ROOT/'weights'/name, sha)


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def make_maskrcnn(finetuned=False, checkpoint=None):
    # Без неявной загрузки backbone и без автоматического изменения числа классов.
    model = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None,
                                 min_size=320, max_size=512, box_score_thresh=.01,
                                 box_nms_thresh=.5, box_detections_per_img=30)
    if checkpoint is None:
        model.load_state_dict(torch.load(weight_path('maskrcnn'), map_location='cpu', weights_only=True))
    if finetuned:
        # person уже есть в COCO. Сохраняем строки background/person предобученных голов.
        old_box, old_mask = model.roi_heads.box_predictor, model.roi_heads.mask_predictor
        box = FastRCNNPredictor(old_box.cls_score.in_features, 2)
        mask = MaskRCNNPredictor(old_mask.conv5_mask.in_channels, 256, 2)
        if checkpoint is None:
            with torch.no_grad():
                box.cls_score.weight.copy_(old_box.cls_score.weight[:2]); box.cls_score.bias.copy_(old_box.cls_score.bias[:2])
                box.bbox_pred.weight.copy_(old_box.bbox_pred.weight[:8]); box.bbox_pred.bias.copy_(old_box.bbox_pred.bias[:8])
                mask.conv5_mask.load_state_dict(old_mask.conv5_mask.state_dict())
                mask.mask_fcn_logits.weight.copy_(old_mask.mask_fcn_logits.weight[:2]); mask.mask_fcn_logits.bias.copy_(old_mask.mask_fcn_logits.bias[:2])
        model.roi_heads.box_predictor, model.roi_heads.mask_predictor = box, mask
    if checkpoint is not None:
        model.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
    return model.to(DEVICE)


@torch.inference_mode()
def predict_maskrcnn(model, item):
    image, _ = read_sample(item)
    model.eval()
    result = model([torch.from_numpy(image.copy()).permute(2,0,1).float().to(DEVICE)/255])[0]
    keep = (result['labels']==1) & (result['scores']>=.01)
    return {'masks':result['masks'][keep,0].cpu().numpy(), 'scores':result['scores'][keep].cpu().numpy(),
            'boxes':result['boxes'][keep].cpu().numpy()}


def predict_yolo(model, item, finetuned=False):
    image, _ = read_sample(item)
    r = model.predict(image[:,:,::-1].copy(), imgsz=320, conf=.01, iou=.5,
                      max_det=30, classes=[0], retina_masks=True, device=DEVICE, verbose=False)[0]
    masks = np.zeros((0,*image.shape[:2]),dtype=np.float32) if r.masks is None else r.masks.data.cpu().numpy().astype(np.float32)
    # retina_masks возвращает бинарные маски в исходных H,W; это не вероятности.
    assert masks.shape[1:] == image.shape[:2]
    return {'masks':masks, 'scores':r.boxes.conf.cpu().numpy(), 'boxes':r.boxes.xyxy.cpu().numpy()}


def raster_boxes(boxes, shape):
    masks = np.zeros((len(boxes), *shape), dtype=bool)
    h,w = shape
    for m,b in zip(masks, boxes):
        x1,y1 = np.floor(b[:2]).astype(int); x2,y2 = np.ceil(b[2:]).astype(int)
        m[max(0,y1):min(h,y2),max(0,x1):min(w,x2)] = True
    return masks


def union_mask(masks, shape):
    return np.any(masks,axis=0) if len(masks) else np.zeros(shape,dtype=bool)


def show_masks(ax, image, masks, title, scores=None):
    ax.imshow(image)
    overlay = np.zeros((*image.shape[:2],4))
    for j,m in enumerate(masks):
        color = plt.get_cmap('tab20')(j%20)
        overlay[np.asarray(m,dtype=bool)] = (*color[:3],.48)
        ys,xs=np.where(m)
        if len(xs): ax.text(xs.mean(),ys.mean(),str(j+1)+(f' / {scores[j]:.2f}' if scores is not None else ''),color='white',fontsize=9,bbox={'facecolor':'black','alpha':.65,'pad':1})
    ax.imshow(overlay); ax.set_title(title); ax.axis('off')


def ap101(tp, scores, n_gt):
    if n_gt == 0: return float('nan')
    order=np.argsort(-np.asarray(scores),kind='stable'); tp=np.asarray(tp,dtype=float)[order]
    if not len(tp): return 0.
    recall=np.cumsum(tp)/n_gt; precision=np.cumsum(tp)/np.arange(1,len(tp)+1)
    return float(np.mean([precision[recall>=r].max(initial=0) for r in np.linspace(0,1,101)]))


def evaluate_predictions(items, predictions, conf=.25, mask_threshold=.5, as_boxes=False):
    # Общий оценщик для обоих методов. GT всегда исходные растровые маски.
    if len(items) != len(predictions) or not items:
        raise ValueError('Для каждого изображения нужен один результат; набор не должен быть пустым')
    tps={t:[] for t in [.5,.75]}; scores=[]; rows=[]; total_gt=0
    for item,p in zip(items,predictions):
        image, ids=read_sample(item); gt=instance_target(ids)['masks'].astype(bool)
        masks = raster_boxes(p['boxes'],ids.shape) if as_boxes else p['masks']>=mask_threshold
        total_gt+=len(gt); scores.extend(p['scores'].tolist())
        for t in tps: tps[t].extend((match_masks(gt,masks,p['scores'],t)>=0).tolist())
        take=p['scores']>=conf; matched=match_masks(gt,masks[take],p['scores'][take],.5)
        tp=int(np.sum(matched>=0)); fp=int(np.sum(matched<0)); fn=len(gt)-tp
        ug,up=union_mask(gt,ids.shape),union_mask(masks[take],ids.shape)
        inter=int(np.logical_and(ug,up).sum()); union=int(np.logical_or(ug,up).sum())
        dice=2*inter/max(1,int(ug.sum()+up.sum()))
        rows.append({'image':item['name'],'TP':tp,'FP':fp,'FN':fn,'foreground_IoU':inter/max(1,union),'foreground_Dice':dice})
    table=pd.DataFrame(rows); tp,fp,fn=table[['TP','FP','FN']].sum()
    metrics={'mask_AP50_above_001':ap101(tps[.5],scores,total_gt),
             'mask_AP75_above_001':ap101(tps[.75],scores,total_gt),
             'precision_025':float(tp/max(1,tp+fp)),'recall_025':float(tp/max(1,tp+fn)),
             'foreground_IoU_macro':float(table.foreground_IoU.mean()),'TP':int(tp),'FP':int(fp),'FN':int(fn)}
    return metrics,table


def predict_subset(key, checkpoint, items):
    start=time.perf_counter()
    model=make_maskrcnn(True,checkpoint) if key=='maskrcnn' else YOLO(str(checkpoint))
    params=sum(p.numel() for p in (model if key=='maskrcnn' else model.model).parameters())
    predictions=[predict_maskrcnn(model,item) if key=='maskrcnn' else predict_yolo(model,item,True) for item in items]
    if DEVICE!='cpu': torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return predictions,elapsed,params

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(min(4,os.cpu_count() or 1))
ROOT=Path(os.environ.get('SEGMENTATION_LAB_ROOT','segmentation_lab_data')).resolve()
ROOT.mkdir(parents=True,exist_ok=True)
seed_all(42)
print('Устройство:',DEVICE,'| torch',torch.__version__,'| torchvision',torchvision.__version__,'| ultralytics',ultralytics.__version__)

SELECTED_MODELS=['maskrcnn','yolo11n_seg']
RUN_BASELINE=True
RUN_PILOT=False
RUN_LONG=False
RUN_TEST=False
ALLOW_CPU_TRAINING=False
TRAIN_CONFIG={'lr':0.0002,'batch_size':2}
PILOT_EPOCHS=2
LONG_EPOCHS=20
HYPOTHESIS=''
PILOT_NOTE=''
FINAL_CONFIG_NOTE=''
TEST_DECISION=''

In [ ]:
split=prepare_data(ROOT)
print({k:len(v) for k,v in split.items()})
image,id_map=read_sample(split['train'][0])
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].imshow(image);axes[0].set_title('Исходное изображение')
axes[1].imshow(id_map,cmap='tab20',interpolation='nearest');axes[1].set_title('Целочисленные ID, 0 — фон')
for ax in axes:ax.axis('off')
plt.tight_layout();plt.show()
print('ID в этом изображении:',np.unique(id_map))

## 1. Реализуйте 1/3: ID → экземпляры и согласованная аугментация

`instance_target(id_map)` получает неотрицательную целочисленную H×W-карту: 0 — фон,
остальные значения — ID. Они могут идти с пропусками: например, 7 и 42.
Верните словарь `masks` uint8 формы N×H×W, `boxes` float32 N×4,
`labels` int64 длины N (везде 1 = person), `area` float32 длины N (число пикселей маски).
Порядок экземпляров — по возрастанию ID. Границы bbox полуоткрытые:
для одного пикселя (x,y) рамка равна [x,y,x+1,y+1].

`hflip_sample` отражает RGB-изображение и все маски слева направо, пересчитывает рамки.
Ширина W: новые x₁=W−x₂, x₂=W−x₁. Исходные массивы менять нельзя.
У пустого изображения должны сохраниться размерности (0,H,W) и (0,4).
Эти функции используются в реальной подготовке обучения, а не только в проверках.

In [ ]:
def instance_target(id_map):
    'Карта H×W, 0=фон → masks[N,H,W], boxes[N,4], labels[N], area[N].'
    # TODO: реализуйте по контракту и проверьте собственным примером.
    raise NotImplementedError("instance_target")

def hflip_sample(image, target):
    'Новое изображение и новый target; не изменять исходные массивы.'
    # TODO: реализуйте по контракту и проверьте собственным примером.
    raise NotImplementedError("hflip_sample")

In [ ]:
"""Small independent fixtures that students can reason about before execution."""
import numpy as np


def check_geometry():
    labels=np.zeros((4,6),dtype=np.uint8); labels[1:3,1:3]=7; labels[3,5]=42
    target=instance_target(labels)
    assert target['masks'].shape==(2,4,6)
    np.testing.assert_array_equal(target['boxes'],[[1,1,3,3],[5,3,6,4]])
    np.testing.assert_array_equal(target['area'],[4,1])
    np.testing.assert_array_equal(target['labels'],[1,1])
    image=np.arange(4*6*3,dtype=np.uint8).reshape(4,6,3)
    new_image,new_target=hflip_sample(image,target)
    np.testing.assert_array_equal(new_target['boxes'],[[3,1,5,3],[0,3,1,4]])
    assert new_target['masks'][1,3,0]==1
    original_image,original_target=hflip_sample(new_image,new_target)
    np.testing.assert_array_equal(original_image,image)
    for k in target: np.testing.assert_array_equal(original_target[k],target[k])
    assert not np.shares_memory(new_target['masks'],target['masks'])
    empty=instance_target(np.zeros((2,3),dtype=np.uint8))
    assert empty['masks'].shape==(0,2,3) and empty['boxes'].shape==(0,4)
    print('Проверено: непоследовательные ID, один пиксель, пустой кадр, двойное отражение.')


def check_metrics():
    a=np.zeros((2,4,4),dtype=bool);a[0,:2,:2]=True;a[1,2:,2:]=True
    b=np.zeros((1,4,4),dtype=bool);b[0,:2,1:3]=True
    np.testing.assert_allclose(mask_iou_matrix(a,b),[[1/3],[0]],atol=1e-7)
    np.testing.assert_allclose(mask_iou_matrix(a,a),np.eye(2))
    assert mask_iou_matrix(a[:0],b).shape==(0,1)
    assert mask_iou_matrix(a,b[:0]).shape==(2,0)
    assert mask_iou_matrix(np.zeros_like(b),np.zeros_like(b))[0,0]==0
    pred=np.stack([a[0],a[0],a[1]])
    np.testing.assert_array_equal(match_masks(a,pred,[.7,.9,.8]),[-1,0,1])
    np.testing.assert_array_equal(match_masks(a,pred,[.9,.9,.8]),[0,-1,1])
    np.testing.assert_array_equal(match_masks(a[:0],pred,[.7,.9,.8]),[-1,-1,-1])
    assert len(match_masks(a,pred[:0],[]))==0
    # Нельзя только argmax по всем GT: лучший уже занят, второй ещё допустим.
    gt=np.zeros((2,1,10),bool);gt[0,0,:6]=1;gt[1,0,4:]=1
    p=np.stack([gt[0],np.ones((1,10),bool)])
    np.testing.assert_array_equal(match_masks(gt,p,[.9,.8],.5),[0,1])
    # У объединённых GT foreground IoU=1, но при пороге .75 нет ни одной пары.
    merged=a.any(0,keepdims=True)
    np.testing.assert_array_equal(match_masks(a,merged,[.9],.75),[-1])
    print('Проверено: IoU=1/3, пустые наборы, дубликаты, равные score, занятый GT и слияние.')


def check_prototypes():
    p=np.array([[[2.,2.],[2.,2.]],[[0.,4.],[0.,4.]]])
    c=np.array([[1.,-1.],[1.,1.]])
    out=assemble_masks(p,c,np.array([[0,0,2,2],[0,0,1,2]]))
    np.testing.assert_allclose(out[0],[[.88079708,.11920292],[.88079708,.11920292]],atol=1e-7)
    np.testing.assert_allclose(out[1,:,1],0)
    assert out.shape==(2,2,2) and np.isfinite(out).all()
    assert assemble_masks(p,np.zeros((0,2)),np.zeros((0,4))).shape==(0,2,2)
    print('Проверено: отрицательный коэффициент, sigmoid после суммы, crop, пустой набор.')

check_geometry()

In [ ]:
target=instance_target(id_map)
flipped,flipped_target=hflip_sample(image,target)
fig,axes=plt.subplots(1,2,figsize=(12,5))
show_masks(axes[0],image,target['masks'],'До отражения')
show_masks(axes[1],flipped,flipped_target['masks'],'После отражения')
plt.tight_layout();plt.show()

**Ваш пример.** Задайте маленькую карту с объектом у левой границы. До запуска
запишите ожидаемые маску и bbox после отражения. Дополните пример двойным отражением.
Почему resize для карты ID должен использовать nearest neighbour?

Ответ: …

## 2. Реализуйте 2/3: IoU масок и уникальное сопоставление

`mask_iou_matrix(a,b)` получает A×H×W и B×H×W масок, возвращает матрицу A×B.
Пустое объединение даёт 0. Проверьте пустой набор масок отдельно от пустой маски.
Для вычисления площадей не используйте умножение uint8: число пикселей может превысить 255.

`match_masks(gt,pred,scores,iou_threshold)` работает с одним классом и изображением.
Возвращает для каждого предсказания индекс уникальной GT-маски либо −1.
Обход — stable сортировка по убыванию score; среди свободных GT выбирайте максимальный
IoU ≥ порога. При равенстве IoU — меньший индекс GT. Результат хранится в **исходном**
порядке предсказаний. Дубликат уже найденного объекта — FP.

Ваши функции использует общий оценщик обеих моделей. Простое совпадение переднего
плана не заменяет сопоставления экземпляров.

In [ ]:
def mask_iou_matrix(a, b):
    'Bool-маски A×H×W, B×H×W → IoU A×B; union=0 даёт 0.'
    # TODO: реализуйте по контракту и проверьте собственным примером.
    raise NotImplementedError("mask_iou_matrix")

def match_masks(gt, pred, scores, iou_threshold=.5):
    'Индекс GT для каждого исходного предсказания либо -1. Один класс.'
    # TODO: реализуйте по контракту и проверьте собственным примером.
    raise NotImplementedError("match_masks")

In [ ]:
check_metrics()

**Ваш пример.** Постройте два GT и три предсказания, одно из которых дублирует объект.
Укажите ожидаемые TP/FP/FN и измените confidence дубликата. Проверьте случай, когда
наиболее похожий GT уже занят, но второй ещё подходит по порогу.

Ответ: …

## 3. Реализуйте 3/3: сборка масок из прототипов

`assemble_masks(P,C,boxes)`: P имеет форму K×H×W, C — N×K, boxes — N×4.
Вычислите N карт логитов как взвешенную сумму прототипов, примените sigmoid к сумме,
затем занулите пиксели вне соответствующей полуоткрытой рамки. Координаты рамок здесь
уже заданы в сетке прототипов; отдельный resize не требуется. Порогование не входит в функцию.
Выход — вещественные вероятности N×H×W. Коэффициенты могут быть отрицательными.

Это контролируемая модель масочной ветви, а не замена внутренней реализации YOLO.
Она позволяет проверить, что вы понимаете механизм общего базиса масок.

In [ ]:
def assemble_masks(prototypes, coefficients, boxes):
    'P[K,H,W], C[N,K], полуоткрытые bbox[N,4] → probabilities[N,H,W].'
    # TODO: реализуйте по контракту и проверьте собственным примером.
    raise NotImplementedError("assemble_masks")

In [ ]:
check_prototypes()

In [ ]:
"""Small interactive teaching stands; all have ordinary function-call fallbacks."""
import ipywidgets as widgets


def overlap_demo(dx=3, background=64):
    gt=np.zeros((background,background),bool);pred=np.zeros_like(gt)
    gt[8:18,8:18]=True;pred[8:18,8+dx:18+dx]=True
    intersection=(gt&pred).sum();iou=intersection/max(1,(gt|pred).sum());dice=2*intersection/max(1,gt.sum()+pred.sum())
    fig,ax=plt.subplots(figsize=(7,4));picture=np.zeros((*gt.shape,3));picture[gt]=(0.2,.55,1);picture[pred]=(1,.4,.15);picture[gt&pred]=(.5,.25,.8)
    ax.imshow(picture);ax.set_title(f'IoU={iou:.3f} · Dice={dice:.3f} · accuracy={(gt==pred).mean():.3f}');ax.axis('off');plt.show()
    return {'IoU':float(iou),'Dice':float(dice),'accuracy':float((gt==pred).mean())}


def merge_demo(merge=True, match_iou=.75):
    gt=np.zeros((2,32,48),bool);gt[0,8:25,6:24]=1;gt[1,8:25,24:42]=1
    pred=gt.any(0,keepdims=True) if merge else gt.copy();scores=np.ones(len(pred))*.9
    matches=match_masks(gt,pred,scores,match_iou);tp=int((matches>=0).sum());image=np.full((32,48,3),225,np.uint8)
    fig,axes=plt.subplots(1,2,figsize=(10,4));show_masks(axes[0],image,gt,'Два GT-экземпляра');show_masks(axes[1],image,pred,f'Предсказано {len(pred)} · TP={tp}, FP={len(pred)-tp}, FN={len(gt)-tp}')
    plt.suptitle('Foreground IoU = 1 в обоих вариантах');plt.tight_layout();plt.show()


def roi_demo(x=.25, y=.5):
    values=np.array([[2.,4.],[6.,8.]])
    weights=np.array([[(1-x)*(1-y),x*(1-y)],[(1-x)*y,x*y]])
    value=float((values*weights).sum());nearest=values[int(y>=.5),int(x>=.5)]
    fig,ax=plt.subplots(figsize=(6,4));ax.set(xlim=(-.15,1.15),ylim=(1.15,-.15),xticks=[0,1],yticks=[0,1])
    for j in range(2):
        for i in range(2):ax.scatter(i,j,s=400,c='#3631ab');ax.text(i,j,str(int(values[j,i])),ha='center',va='center',color='white')
    ax.scatter(x,y,c='orange',s=180);ax.grid(alpha=.2);ax.set_title(f'Билинейно: {value:.3f} · ближайший узел: {nearest:.1f}')
    plt.show();return value


def prototype_demo(second=-1., threshold=.5):
    y,x=np.mgrid[:48,:64];whole=((x>8)&(x<55)&(y>6)&(y<42)).astype(float)*4
    right=((x>=32)&(x<55)&(y>6)&(y<42)).astype(float)*8
    p=np.stack([whole,right,np.ones_like(whole)]);c=np.array([[1.,second,-2.]])
    masks=assemble_masks(p,c,np.array([[8,6,56,43]]))
    fig,axes=plt.subplots(1,4,figsize=(14,3))
    for ax,img,title in zip(axes,[whole,right,masks[0],masks[0]>=threshold],['Прототип 1','Прототип 2',f'sigmoid(P₁ {second:+.1f}P₂ − 2)',f'Порог {threshold:.2f}']):
        ax.imshow(img,cmap='viridis');ax.set_title(title);ax.axis('off')
    plt.tight_layout();plt.show()
    return masks


def threshold_demo(item, prediction, confidence=.5, mask_threshold=.5):
    image,ids=read_sample(item);gt=instance_target(ids)['masks'];take=prediction['scores']>=confidence
    masks=prediction['masks'][take]>=mask_threshold;boxes=raster_boxes(prediction['boxes'][take],ids.shape)
    matches=match_masks(gt,masks,prediction['scores'][take],.5);tp=int((matches>=0).sum())
    fig,axes=plt.subplots(1,3,figsize=(16,5))
    show_masks(axes[0],image,gt,'GT: видимые части человека')
    show_masks(axes[1],image,boxes,'Заполненные рамки',prediction['scores'][take])
    show_masks(axes[2],image,masks,f'Mask R-CNN · TP {tp}, FP {len(masks)-tp}, FN {len(gt)-tp}',prediction['scores'][take])
    plt.suptitle(f'confidence ≥ {confidence:.2f}; вероятность пикселя ≥ {mask_threshold:.2f}')
    plt.tight_layout();plt.show()


def real_comparison(item, maskrcnn_prediction, yolo_prediction):
    image,ids=read_sample(item);gt=instance_target(ids)['masks']
    fig,axes=plt.subplots(1,3,figsize=(16,5));show_masks(axes[0],image,gt,'GT')
    for ax,p,title in [(axes[1],maskrcnn_prediction,'Mask R-CNN'),(axes[2],yolo_prediction,'YOLO11n-seg')]:
        take=p['scores']>=.25;show_masks(ax,image,p['masks'][take]>=.5,title,p['scores'][take])
    plt.tight_layout();plt.show()

prototype_demo(second=-1.,threshold=.5)

**Ваш пример.** Возьмите две карты 2×2 и коэффициенты [1,−1]. Посчитайте один пиксель
вручную, затем покажите, почему sigmoid(P₁−P₂) отличается от sigmoid(P₁)−sigmoid(P₂).

Ответ: …

In [ ]:
"""Training and evaluation cells, embedded in both student and instructor notebooks."""
import inspect
from torch.utils.data import Dataset, DataLoader
from ultralytics.data.converter import merge_multi_segment


class PedestrianDataset(Dataset):
    def __init__(self, items, training=False): self.items,self.training=items,training
    def __len__(self): return len(self.items)
    def __getitem__(self,index):
        image,labels=read_sample(self.items[index]);target=instance_target(labels)
        if self.training and random.random()<.5: image,target=hflip_sample(image,target)
        target={k:torch.as_tensor(v) for k,v in target.items()}
        target['image_id']=torch.tensor(index);target['iscrowd']=torch.zeros(len(target['labels']),dtype=torch.int64)
        return torch.from_numpy(image.copy()).permute(2,0,1).float()/255,target


def collate_samples(batch): return tuple(zip(*batch))


def export_yolo(split, folder):
    """YOLO polygons: соединяем компоненты; отверстия не поддерживаются этим форматом."""
    folder=Path(folder);rows=[]
    for part in ('train','val'):  # test остаётся только в исходном растровом формате.
        (folder/'images'/part).mkdir(parents=True,exist_ok=True)
        (folder/'labels'/part).mkdir(parents=True,exist_ok=True)
        for item in split[part]:
            image,ids=read_sample(item);target=instance_target(ids);h,w=ids.shape;lines=[]
            destination=folder/'images'/part/item['name']
            if not destination.exists(): shutil.copy2(item['image'],destination)
            for j,m in enumerate(target['masks']):
                contours,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                segments=[c.reshape(-1,2) for c in contours if len(c)>=3]
                if not segments:
                    raise ValueError(f"Маска {item['name']}:{j} не представима полигоном: разберите её отдельно")
                points=np.concatenate(merge_multi_segment(segments)) if len(segments)>1 else segments[0]
                roundtrip=np.zeros((h,w),np.uint8);cv2.fillPoly(roundtrip,[points.astype(np.int32)],1)
                iou=float(np.logical_and(m,roundtrip).sum()/max(1,np.logical_or(m,roundtrip).sum()))
                rows.append({'split':part,'image':item['name'],'instance':j,'polygon_IoU':iou,'components':len(segments)})
                norm=points/np.array([w,h]);lines.append('0 '+' '.join(f'{v:.8f}' for v in norm.ravel()))
            (folder/'labels'/part/(Path(item['name']).stem+'.txt')).write_text('\n'.join(lines)+'\n')
    import yaml
    config={'path':str(folder.resolve()),'train':'images/train','val':'images/val','names':{0:'person'}}
    (folder/'data.yaml').write_text(yaml.safe_dump(config,sort_keys=False))
    audit=pd.DataFrame(rows);audit.to_csv(folder/'polygon_audit.csv',index=False)
    return folder/'data.yaml',audit


def functions_fingerprint():
    # Не зависит от пути ячейки; изменение реализации студента меняет ключ результата.
    def code_payload(code):
        return {'bytecode':code.co_code.hex(),'names':code.co_names,
                'constants':[code_payload(c) if isinstance(c,type(code)) else repr(c) for c in code.co_consts]}
    return {f.__name__:hashlib.sha256(json.dumps(code_payload(f.__code__),sort_keys=True).encode()).hexdigest()
            for f in [instance_target,hflip_sample,mask_iou_matrix,match_masks,assemble_masks]}


def protocol_for(stage, epochs, config, split):
    return {'schema':1,'stage':stage,'epochs':int(epochs),'config':dict(config),'seed':42,
            'data_sha256':DATA_SHA,'split':{k:[i['name'] for i in v] for k,v in split.items()},
            'weights':{k:v[2] for k,v in WEIGHT_SOURCES.items()},'functions':functions_fingerprint(),
            'versions':{'torch':torch.__version__,'torchvision':torchvision.__version__,'ultralytics':ultralytics.__version__},
            'evaluation':{'score_floor':.01,'confidence':.25,'mask_threshold':.5,'max_det':30,'checkpoint':'last'}}


def train_maskrcnn(items, epochs, config, folder):
    seed_all();model=make_maskrcnn(finetuned=True)
    loader=DataLoader(PedestrianDataset(items,True),batch_size=config['batch_size'],shuffle=True,
                      num_workers=0,collate_fn=collate_samples,generator=torch.Generator().manual_seed(42))
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=config['lr'],weight_decay=.0001)
    history=[]
    for epoch in range(epochs):
        model.train()
        # Сохраняем COCO running statistics: batch=2 слишком мал для их оценки.
        for m in model.modules():
            if isinstance(m,torch.nn.modules.batchnorm._BatchNorm): m.eval()
        losses=[];start=time.perf_counter()
        for images,targets in loader:
            images=[im.to(DEVICE) for im in images];targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
            values=model(images,targets);loss=sum(values.values())
            if not torch.isfinite(loss): raise FloatingPointError('Нечисловая функция потерь')
            optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step();losses.append({k:float(v.detach()) for k,v in values.items()})
        row={'epoch':epoch+1,'seconds':time.perf_counter()-start,**pd.DataFrame(losses).mean().to_dict()}
        history.append(row);print('Mask R-CNN:',epoch+1,'/',epochs,'loss',round(sum(row[k] for k in row if k.startswith('loss')),4),flush=True)
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        torch.save(model.cpu().state_dict(),folder/'last.pth');model.to(DEVICE)
    del model,optimizer;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return folder/'last.pth'


def train_yolo(data_yaml, epochs, config, folder):
    seed_all();model=YOLO(str(weight_path('yolo11n_seg')))
    model.train(data=str(data_yaml),epochs=epochs,imgsz=320,batch=config['batch_size'],device=DEVICE,
                project=str(folder),name='training',exist_ok=False,workers=0,seed=42,deterministic=True,
                optimizer='AdamW',lr0=config['lr'],lrf=1.,weight_decay=.0001,warmup_epochs=0.,nbs=config['batch_size'],
                cos_lr=False,amp=False,cache=False,plots=False,verbose=False,save=True,save_period=-1,
                patience=epochs+1,freeze=None,mosaic=0.,mixup=0.,copy_paste=0.,close_mosaic=0,
                hsv_h=0.,hsv_s=0.,hsv_v=0.,degrees=0.,translate=0.,scale=0.,shear=0.,perspective=0.,
                flipud=0.,fliplr=.5,mask_ratio=1,overlap_mask=False)
    checkpoint=Path(model.trainer.save_dir)/'weights'/'last.pt'
    shutil.copy2(checkpoint,folder/'last.pt')
    shutil.copy2(Path(model.trainer.save_dir)/'results.csv',folder/'history.csv')
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return folder/'last.pt'


def run_series(stage, epochs, config, split):
    if DEVICE=='cpu' and not ALLOW_CPU_TRAINING:
        raise RuntimeError('Для обучения выберите GPU. Для короткой технической пробы можно явно разрешить CPU.')
    if config['batch_size']<1 or config['lr']<=0: raise ValueError('Проверьте batch_size и lr')
    protocol=protocol_for(stage,epochs,config,split)
    key=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:12]
    directory=ROOT/'runs'/f'{stage}_{key}';directory.mkdir(parents=True,exist_ok=True)
    data_yaml,audit=export_yolo(split,ROOT/'yolo_polygons'/key)
    print('Минимальный IoU после преобразования в полигон:',round(audit.polygon_IoU.min(),4),flush=True)
    records=[]
    for name in SELECTED_MODELS:
        folder=directory/name;report=folder/'record.json'
        if report.exists():
            record=json.loads(report.read_text())
            assert record['protocol']==protocol and sha256_file(record['checkpoint'])==record['checkpoint_sha256']
            records.append(record);print('Готовый запуск:',name);continue
        if folder.exists() and any(folder.iterdir()):
            raise RuntimeError(f'Незавершённый запуск в {folder}. Сохраните его отдельно перед повтором; он не будет выдан за готовый.')
        folder.mkdir(parents=True,exist_ok=True);start=time.perf_counter()
        checkpoint=train_maskrcnn(split['train'],epochs,config,folder) if name=='maskrcnn' else train_yolo(data_yaml,epochs,config,folder)
        train_seconds=time.perf_counter()-start
        predictions,seconds,params=predict_subset(name,checkpoint,split['val'])
        metrics,errors=evaluate_predictions(split['val'],predictions)
        errors.assign(errors=errors.FP+errors.FN).sort_values(['errors','image'],ascending=[False,True]).to_csv(folder/'val_errors.csv',index=False)
        record={'model':name,'stage':stage,'protocol':protocol,'checkpoint':str(checkpoint.resolve()),
                'checkpoint_sha256':sha256_file(checkpoint),'parameters':params,'train_seconds':train_seconds,
                'val_seconds_including_load':seconds,'metrics':metrics}
        report.write_text(json.dumps(record,indent=2));records.append(record)
    (directory/'protocol.json').write_text(json.dumps(protocol,indent=2))
    return records


def records_table(records):
    return pd.DataFrame([{'model':r['model'],'stage':r['stage'],'parameters':r['parameters'],
                         'train_minutes':r['train_seconds']/60,'val_seconds_including_load':r['val_seconds_including_load'],**r['metrics']} for r in records])


def plot_history(records):
    fig,axes=plt.subplots(1,len(records),figsize=(6*len(records),4),squeeze=False)
    for ax,r in zip(axes[0],records):
        history=pd.read_csv(Path(r['checkpoint']).parent/'history.csv');history.columns=history.columns.str.strip()
        columns=[c for c in history if c.startswith('loss') or c.startswith('train/')]
        for c in columns:ax.plot(history['epoch'],history[c],label=c)
        ax.set(title=r['model'],xlabel='Эпоха',ylabel='Компонента loss');ax.legend(fontsize=8)
    plt.tight_layout();plt.show()


def compare_frames(records, items):
    predictions={r['model']:predict_subset(r['model'],r['checkpoint'],items)[0] for r in records}
    for i,item in enumerate(items):
        image,ids=read_sample(item);gt=instance_target(ids)['masks']
        fig,axes=plt.subplots(1,len(records)+1,figsize=(6*(len(records)+1),5))
        show_masks(axes[0],image,gt,'GT · '+item['name'])
        for ax,r in zip(axes[1:],records):
            p=predictions[r['model']][i];take=p['scores']>=.25
            show_masks(ax,image,p['masks'][take]>=.5,r['model'],p['scores'][take])
        plt.tight_layout();plt.show()
    return predictions


def final_test(records, split, decision):
    if not records or {r['model'] for r in records}!=set(SELECTED_MODELS) or any(r['stage']!='long' for r in records):
        raise ValueError('Нужны завершённые длинные запуски обеих моделей')
    if len(decision.strip())<30: raise ValueError('Запишите обоснование решения по val до открытия test')
    protocol={'records':records,'decision':decision,'test_names':[i['name'] for i in split['test']]}
    lock=ROOT/'final_test_protocol.json';result=ROOT/'final_test_results.csv'
    if lock.exists():
        if json.loads(lock.read_text())!=protocol:raise RuntimeError('Test уже открыт с другим протоколом. Не используйте его для настройки.')
        if result.exists():return pd.read_csv(result)
    else:lock.write_text(json.dumps(protocol,indent=2))
    rows=[]
    for r in records:
        if sha256_file(r['checkpoint'])!=r['checkpoint_sha256']:raise ValueError('Checkpoint изменён')
        predictions,seconds,_=predict_subset(r['model'],r['checkpoint'],split['test'])
        metrics,_=evaluate_predictions(split['test'],predictions)
        rows.append({'model':r['model'],'test_seconds_including_load':seconds,**metrics})
    table=pd.DataFrame(rows);table.to_csv(result,index=False);return table

## 4. Подготовка данных и ограничения формата

Mask R-CNN получает исходные растровые маски. YOLO обучается по полигонам:
внешние контуры объединяются для разорванных частей, отверстия не сохраняются.
Проверьте `polygon_audit.csv`: насколько после обратной растеризации изменилась разметка?
Это дополнительное различие рецептов. **Val/test обеих моделей оцениваются по исходным
растровым GT**, а не по полигонам YOLO. Внутренние метрики его тренера не являются итоговой таблицей.

In [ ]:
data_yaml,polygon_audit=export_yolo(split,ROOT/'yolo_preview')
display(polygon_audit.groupby('split').polygon_IoU.agg(['count','min','mean']))
display(polygon_audit.sort_values('polygon_IoU').head(5))
counts=[]
for part in ['train','val']:
    for item in split[part]:
        _,ids=read_sample(item);t=instance_target(ids)
        counts.append({'split':part,'image':item['name'],'instances':len(t['masks']),'pixels':float(t['area'].sum())})
display(pd.DataFrame(counts).groupby('split').agg(images=('image','count'),instances=('instances','sum')))

Запишите, какой train-объект имеет худшее приближение полигоном. Откройте его ID-карту,
найдите отверстие или разорванную часть и объясните причину отличия. Не исправляйте val/test
по ошибкам своей модели. Какие ещё ограничения есть у этого небольшого набора?

Ответ: …

## 5. Исходные COCO-модели и рамка вместо маски

Сначала оценим обе исходные модели на 25 val-изображениях. Дополнительная строка
«Mask R-CNN: заполненные рамки» использует **те же экземпляры, confidence и рамки**, но
заменяет каждый силуэт прямоугольником. Это контроль важности формы, а не третья обученная модель.

AP50 и AP75 здесь — 101-точечная интерполированная AP одного класса при IoU 0.5/0.75,
score ≥ 0.01 и максимум 30 предсказаний на кадр. Это **не COCO AP@[.50:.95]**.
Precision/recall считаются при confidence=0.25, mask threshold=0.5 и match IoU=0.5.
Foreground IoU — среднее по изображениям для объединения масок, без разделения экземпляров.

In [ ]:
baseline_rows=[]
if RUN_BASELINE:
    for name in SELECTED_MODELS:
        model=make_maskrcnn() if name=='maskrcnn' else YOLO(str(weight_path(name)))
        predictions=[predict_maskrcnn(model,item) if name=='maskrcnn' else predict_yolo(model,item) for item in split['val']]
        metrics,_=evaluate_predictions(split['val'],predictions)
        baseline_rows.append({'model':name,**metrics})
        if name=='maskrcnn':
            rectangle_metrics,_=evaluate_predictions(split['val'],predictions,as_boxes=True)
            baseline_rows.append({'model':'Mask R-CNN: заполненные рамки',**rectangle_metrics})
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    baseline_table=pd.DataFrame(baseline_rows);baseline_table.to_csv(ROOT/'baseline_val.csv',index=False);display(baseline_table)
else:print('Исходное сравнение пропущено. Вернитесь к нему перед выводами о дообучении.')

## 6. Пилоты на занятии: две модели × две эпохи

Проверьте реализации и заполните `HYPOTHESIS`. Затем включите `RUN_PILOT=True`
в конфигурации и выполните ячейку ниже. Гипотеза должна указывать механизм и ожидаемую
ошибку, а не просто «вторая модель лучше».

Общий учебный бюджет: AdamW, lr=0.0002, batch=2, только горизонтальное отражение train,
320 пикселей, без mosaic и смешивания изображений. Все параметры сети обучаются;
у Mask R-CNN статистики BatchNorm фиксируются из-за малого batch. YOLO использует
свой нормализатор и обучающие потери. Для нового единственного класса его библиотека
перестраивает классификационный выход; у Mask R-CNN сохраняются COCO-строки background/person.

Смотрите loss, предсказания и ошибки на val. Не сравнивайте абсолютные суммы loss разных
архитектур. После пилотов обоснуйте сохранение настройки или изменение **одного** параметра.

In [ ]:
pilot_records=[]
if RUN_PILOT:
    if len(HYPOTHESIS.strip())<30:raise ValueError('Сначала запишите содержательную гипотезу в конфигурации')
    pilot_records=run_series('pilot',PILOT_EPOCHS,TRAIN_CONFIG,split)
    display(records_table(pilot_records));plot_history(pilot_records)
else:print('RUN_PILOT=False: обучение не запущено.')

## 7. После занятия: две модели × 20 эпох

Заполните `PILOT_NOTE` наблюдениями, а `FINAL_CONFIG_NOTE` — решением по настройке.
Включите `RUN_LONG=True`. Обе серии начинаются **заново от COCO**, а не продолжают пилоты.
Используем последнюю эпоху, без выбора лучшего checkpoint по test. Двадцать эпох —
учебный бюджет, не обещание сходимости. Длинные серии не нужно ждать на занятии.
При нехватке памяти уменьшайте batch для обеих моделей и фиксируйте изменение.

In [ ]:
long_records=[]
if RUN_LONG:
    if min(len(PILOT_NOTE.strip()),len(FINAL_CONFIG_NOTE.strip()))<30:raise ValueError('Запишите наблюдения пилота и решение по настройке')
    long_records=run_series('long',LONG_EPOCHS,TRAIN_CONFIG,split)
    display(records_table(long_records));plot_history(long_records)
else:print('RUN_LONG=False: длинное обучение не запущено.')

## 8. Общая таблица и одинаковые кадры

Таблица показывает AP50/AP75, precision/recall, foreground IoU, параметры и время.
Время val включает загрузку модели и обработку всех изображений; это не чистая задержка сети.
В сравнении времени запишите GPU/CPU, версии и размер входа.

По `val_errors.csv` автоматически выберите кадр с наибольшим FP+FN, ещё один с тонкими
частями или перекрытием и обычный удачный кадр. Используйте эти же изображения для обеих моделей.

In [ ]:
active_records=long_records or pilot_records
if active_records:
    table=records_table(active_records);display(table);table.to_csv(ROOT/'comparison_val.csv',index=False)
    selected=[]
    for r in active_records:
        errors=pd.read_csv(Path(r['checkpoint']).parent/'val_errors.csv')
        name=errors.iloc[0]['image']
        if name not in selected:selected.append(name)
    for item in split['val']:
        if len(selected)>=3:break
        if item['name'] not in selected:selected.append(item['name'])
    by_name={i['name']:i for i in split['val']}
    selected_items=[by_name[name] for name in selected[:3]]
    compared_predictions=compare_frames(active_records,selected_items)
else:print('После завершения пилотов или длинных серий появится сравнение.')

### Пять выводов, опирающихся на результаты

1. Где ваши контрольные примеры обнаруживали ошибки в геометрии, matching и прототипах?
2. Насколько заполненная рамка хуже силуэта по AP75? Изменилась ли эта разница в foreground IoU?
3. Что показали пилоты и почему вы сохранили или изменили настройку? Улучшились ли модели после длинной серии относительно COCO?
4. Для 2–3 одинаковых кадров разделите ошибки локализации, границы, слияния/разделения и неполноту GT.
5. Какую модель выбрали бы для этой задачи с учётом качества и затрат? Какие различия опыта мешают объяснить всё одним устройством масочной головы?

Ответы: …

## 9. Зафиксировать решение и один раз открыть test

Заполните `TEST_DECISION` по val, затем включите `RUN_TEST=True`.
Сначала сохраняется протокол с настройками, списком изображений и SHA-256 checkpoint;
только после этого выполняется оценка. Повторный запуск с тем же протоколом читает
сохранённую таблицу. Другой протокол блокируется. Это помощь в дисциплине опыта,
а не техническая защита от доступа к данным. Test не используем для следующего подбора.

In [ ]:
if RUN_TEST:display(final_test(long_records,split,TEST_DECISION))
else:print('RUN_TEST=False: test не оценивался.')

## Что сдавать

- Выполненный ноутбук с тремя реализациями, пройденными проверками и собственными примерами.
- `baseline_val.csv`, `comparison_val.csv`, `final_test_results.csv`, протоколы и журналы обучения обеих моделей.
- Два пилота по 2 эпохи, две длинные серии по 20 эпох, кривые и обоснование настройки.
- 2–3 общих изображения с наложенными масками и пять предметных выводов.

Веса и весь архив данных сдавать не требуется. Оцениваются корректность и анализ,
а не победа в метрике или мощность видеокарты. Дополнительно можно сравнить один из
результатов с маской по подсказке SAM; такой опыт требует описать источник подсказок
и не заменяет сравнение двух автоматических моделей.